In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
# --------------------- Import VQNiche ---------------------
from vqniche.utils.parse_test_configs import *
from vqniche.initializers.initialize import *
from vqniche.utils.type_conversions import *
from vqniche.plotting import *

/software/cellgen/team361/am84/envs/vqniche-reproducibility/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/software/cellgen/team361/am84/envs/vqniche-reproducibility/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/software/cellgen/team361/am84/envs/vqniche-reproducibility/lib/python3.10/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain

In [3]:
# --------------------- Import Libraries ---------------------
import os
import copy
import sys
import yaml
import pickle
from pathlib import Path
from dataclasses import dataclass

import scanpy as sc
import anndata as ad
import squidpy as sq

import numpy as np
import networkx as nx
import scipy.sparse as sp
from scipy.stats import pearsonr

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import pytorch_lightning as pl
import torch_geometric.transforms as T
from torch_geometric.data import Batch
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_adj
from torch_geometric.loader import DataLoader as BatchBuilder

# --------------------- Display Settings ---------------------
# display setting all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
mmb0_239b_1p_random_file = Path("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/reproducibility/config/train_model/mmb0-239b_1p_3-test-patch-split_vqniche_graphsage.yaml")
with open(mmb0_239b_1p_random_file, 'r') as f:
    mmb0_239b_1p_random_cfg = yaml.safe_load(f)
    
print("Loading dataset...")
# Initialize dataset blob (this loads the raw data and applies transforms)
dataset_blob = initialize_dataset_blob(
    mmb0_239b_1p_random_cfg
)

print("Initializing data batch...")
# Initialize data batch (this loads the specific batch/tissue section)
big_brain = initialize_databatch(
    config=mmb0_239b_1p_random_cfg,
    dataset_blob=dataset_blob
)

Loading dataset...
Initializing data batch...
SubsetHVG: Subsetted data to 1000 features.
Setting section-level conditioning features for encoder.
Setting section-level conditioning features for attribute decoder.
SubsetHVG: Subsetted data to 1000 features.
Setting section-level conditioning features for encoder.
Setting section-level conditioning features for attribute decoder.
SubsetHVG: Subsetted data to 1000 features.
Setting section-level conditioning features for encoder.
Setting section-level conditioning features for attribute decoder.
SubsetHVG: Subsetted data to 1000 features.
Setting section-level conditioning features for encoder.
Setting section-level conditioning features for attribute decoder.
SubsetHVG: Subsetted data to 1000 features.
Setting section-level conditioning features for encoder.
Setting section-level conditioning features for attribute decoder.
SubsetHVG: Subsetted data to 1000 features.
Setting section-level conditioning features for encoder.
Setting secti

In [7]:
del big_brain

In [4]:
mmb0_4b_1p_3_test_patch_file = Path("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/reproducibility/config/train_model/mmb0-4b_1p_3-test-patch-split_vqniche_graphsage.yaml")
with open(mmb0_4b_1p_3_test_patch_file, 'r') as f:
    mmb0_4b_1p_3_test_patch_cfg = yaml.safe_load(f)
    
print("Loading dataset...")
# Initialize dataset blob (this loads the raw data and applies transforms)
dataset_blob = initialize_dataset_blob(
    mmb0_4b_1p_3_test_patch_cfg
)

print("Initializing data batch...")
# Initialize data batch (this loads the specific batch/tissue section)
brain = initialize_databatch(
    config=mmb0_4b_1p_3_test_patch_cfg,
    dataset_blob=dataset_blob
)

Loading dataset...
Initializing data batch...
SubsetHVG: Subsetted data to 1000 features.
Setting section-level conditioning features for encoder.
Setting section-level conditioning features for attribute decoder.
SubsetHVG: Subsetted data to 1000 features.
Setting section-level conditioning features for encoder.
Setting section-level conditioning features for attribute decoder.
SubsetHVG: Subsetted data to 1000 features.
Setting section-level conditioning features for encoder.
Setting section-level conditioning features for attribute decoder.
SubsetHVG: Subsetted data to 1000 features.
Setting section-level conditioning features for encoder.
Setting section-level conditioning features for attribute decoder.
DataBatch(y_cell_types=[48156, 23], y_niche_types=[48156, 4], xy_coordinates=[48156, 2], cell_id=[4], dataset_id=[4], tissue=[4], species=[4], adata_batch_id=[4], x=[48156, 1000], y=[48156, 23], edge_index=[2, 497936], encoder_conditions=[48156, 13], encoder_condition_dim=13, spati

In [6]:
brain_adj = edge_index_to_adjacency_tensor(brain.edge_index)
brain_G = adjacency_tensor_to_networkx(brain_adj)

In [7]:
print(brain_G)

Graph with 48156 nodes and 273046 edges


In [15]:
brain_cc = [brain_G.subgraph(c).copy() for c in nx.connected_components(brain_G)]

In [17]:
for G in brain_cc:
    print(G)
    print(nx.diameter(G))

Graph with 3514 nodes and 19928 edges
52
Graph with 5985 nodes and 34114 edges
80
Graph with 8612 nodes and 48991 edges
93
Graph with 31 nodes and 187 edges
4
Graph with 14913 nodes and 84321 edges
127
Graph with 15101 nodes and 85505 edges
129


In [7]:
xhs1000_39b_1p_oriented_7_3_test_patch_file = Path("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/logs/xhs1000-39b_1p/sweep/VQNiche/batch=[2, 11, 9, 28, 29, 32, 12]/spatial_n_neighs_8/seed/20250923-222431/wandb/offline-run-20250923_222431-mank50dk/files/user_specified_config.yaml")
with open(xhs1000_39b_1p_oriented_7_3_test_patch_file, 'r') as f:
    xhs1000_39b_1p_oriented_7_3_test_patch_cfg = yaml.safe_load(f)
    
print("Loading dataset...")
# Initialize dataset blob (this loads the raw data and applies transforms)
dataset_blob = initialize_dataset_blob(
    xhs1000_39b_1p_oriented_7_3_test_patch_cfg
)

print("Initializing data batch...")
# Initialize data batch (this loads the specific batch/tissue section)
skin = initialize_databatch(
    config=xhs1000_39b_1p_oriented_7_3_test_patch_cfg,
    dataset_blob=dataset_blob
)

Loading dataset...
Initializing data batch...
SubsetHVG: Subsetted data to 1000 features.
SubsetHVG: Subsetted data to 1000 features.
SubsetHVG: Subsetted data to 1000 features.
SubsetHVG: Subsetted data to 1000 features.
SubsetHVG: Subsetted data to 1000 features.
SubsetHVG: Subsetted data to 1000 features.
SubsetHVG: Subsetted data to 1000 features.
DataBatch(y_cell_types=[110278, 41], y_niche_types=[110278, 15], xy_coordinates=[110278, 2], cell_id=[7], dataset_id=[7], tissue=[7], species=[7], adata_batch_id=[7], x=[110278, 1000], y=[110278, 41], edge_index=[2, 1141520], encoder_condition_dim=0, spatial_prior_feature_dim=0, attr_decoder_condition_dim=0, adj_decoder_condition_dim=0, num_features=1000, num_classes=41, num_nodes=110278, num_edges=[7], train_mask=[110278], val_mask=[110278], test_mask=[110278], batch=[110278], ptr=[8], adata_batch_ids=[110278])
Batch ID(s): [2, 11, 9, 28, 29, 32, 12]
Data Batch: DataBatch(y_cell_types=[110278, 41], y_niche_types=[110278, 15], xy_coordina

In [9]:
xhk1020_CV1_CV2_5b_1p_all_sections_all_cells_file = Path("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/logs/xhk1020-CV1-CV2-5b_1p/sweep/VQNiche/batch=[0, 1, 2, 3, 4]/spatial_n_neighs_8/seed/20250918-142428/wandb/run-20250918_142429-aux6obdu/files/user_specified_config.yaml")
with open(xhk1020_CV1_CV2_5b_1p_all_sections_all_cells_file, 'r') as f:
    xhk1020_CV1_CV2_5b_1p_all_sections_all_cells_cfg = yaml.safe_load(f)
    
print("Loading dataset...")
# Initialize dataset blob (this loads the raw data and applies transforms)
dataset_blob = initialize_dataset_blob(
    xhk1020_CV1_CV2_5b_1p_all_sections_all_cells_cfg
)

print("Initializing data batch...")
# Initialize data batch (this loads the specific batch/tissue section)
kidney = initialize_databatch(
    config=xhk1020_CV1_CV2_5b_1p_all_sections_all_cells_cfg,
    dataset_blob=dataset_blob
)

Loading dataset...
Initializing data batch...
SubsetHVG: Subsetted data to 1000 features.
SubsetHVG: Subsetted data to 1000 features.
SubsetHVG: Subsetted data to 1000 features.
SubsetHVG: Subsetted data to 1000 features.
SubsetHVG: Subsetted data to 1000 features.
DataBatch(y_grade=[260193, 2], y_clinical_risk=[260193, 2], xy_coordinates=[260193, 2], cell_id=[5], dataset_id=[5], tissue=[5], species=[5], adata_batch_id=[5], x=[260193, 1000], y=[260193, 2], edge_index=[2, 2673293], encoder_condition_dim=0, spatial_prior_feature_dim=0, attr_decoder_condition_dim=0, adj_decoder_condition_dim=0, num_features=1000, num_classes=2, num_nodes=260193, num_edges=[5], train_mask=[260193], val_mask=[260193], test_mask=[260193], batch=[260193], ptr=[6], adata_batch_ids=[260193])
Batch ID(s): [0, 1, 2, 3, 4]
Data Batch: DataBatch(y_grade=[260193, 2], y_clinical_risk=[260193, 2], xy_coordinates=[260193, 2], cell_id=[5], dataset_id=[5], tissue=[5], species=[5], adata_batch_id=[5], x=[260193, 1000], y=

In [20]:
import numpy as np
import torch

def summarize_databatch(data_batch):
    """
    Summarize key statistics of a PyTorch Geometric DataBatch-like object.

    Parameters
    ----------
    data_batch : torch_geometric.data.Data or DataBatch
        The dataset object containing attributes like x, y, edge_index, etc.

    Returns
    -------
    stats : dict
        Dictionary of dataset statistics.
    """
    stats = {}

    # --- Basic sizes ---
    stats["Cells"] = data_batch.x.shape[0]
    stats["Edges"] = data_batch.edge_index.shape[1]
    stats["Genes"] = data_batch.x.shape[1]

    # --- Graph / batch info ---
    if hasattr(data_batch, "batch"):
        stats["Num Sections"] = int(data_batch.batch.max().item() + 1)

    # --- Label distribution (if categorical) ---
    if hasattr(data_batch, "y_cell_types"):
        stats["Number of Cell Types"] = data_batch.y_cell_types.shape[1]
    else:
        stats["Number of Cell Types"] = "NA"
        
    return stats


In [30]:
import pandas as pd

def collect_dataset_stats(datasets, dataset_names=None):
    """
    Collect summary stats for multiple datasets and convert to a pandas DataFrame.

    Parameters
    ----------
    datasets : list
        List of PyG Data or DataBatch objects.
    dataset_names : list[str], optional
        Names for each dataset. If None, indices are used.

    Returns
    -------
    df : pd.DataFrame
        A DataFrame where each row is a dataset and columns are statistics.
    """
    all_stats = []
    if dataset_names is None:
        dataset_names = [f"dataset_{i}" for i in range(len(datasets))]

    for name, ds in zip(dataset_names, datasets):
        stats = summarize_databatch(ds)
        stats["dataset_name"] = name
        all_stats.append(stats)

    # convert to dataframe
    df = pd.DataFrame(all_stats).set_index("dataset_name")
    return df


In [31]:
data_batches = [brain, skin, kidney]
names = ['Brain (Mouse)', 'Skin (Human)', 'Kidney (Human)']

df_stats = collect_dataset_stats(data_batches, dataset_names=names)
display(df_stats)

,Cells,Edges,Genes,Num Sections,Number of Cell Types
dataset_name,,,,,
Brain (Mouse),48156,497936,1000,4,23
Skin (Human),110278,1141520,1000,7,41
Kidney (Human),260193,2673293,1000,5,NA


In [27]:
# Save DataFrame to LaTeX
def save_stats_to_latex(df, filename="dataset_statistics.tex"):
    """
    Save dataset stats DataFrame to a LaTeX table.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame of dataset stats (from collect_dataset_stats).
    filename : str
        Output filename for the LaTeX table.
    """
    # convert to LaTeX with some formatting
    latex_str = df.to_latex(
        escape=False,      # allow LaTeX in content if present
        index=True,        # keep dataset names
        longtable=False,    # use longtable for big tables
        multicolumn=False,  # align multi-level cols
        multicolumn_format="c",
        caption="Summary statistics of all datasets.",
        label="tab:dataset_stats"
    )

    with open(filename, "w") as f:
        f.write(latex_str)

    print(f"LaTeX table written to {filename}")

fname = "/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/paper/tables/dataset_statistics.tex"
save_stats_to_latex(df_stats, fname)

LaTeX table written to /lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/paper/tables/dataset_statistics.tex
